# 🦿 Jev API → Gymnasium BipedalWalker Controller

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/simple-jev/blob/main/notebooks/Jev_API_BipedalWalker_Controller_Colab.ipynb)

This notebook uses **TypeSafe Jev via an API key** to make real control decisions for `BipedalWalker-v3`.

### API key
Create/get your TypeSafe API key at **https://typesafe.ai** (the developer console is at `console.typesafe.ai`), then add it to Colab Secrets as:

`TYPESAFE_API_KEY`

### Why Jev chooses motor plans instead of generating four floats
Jev exposes typed decisions such as **Choice**, not arbitrary numeric regression. This notebook therefore:
1. reads the 24-value Gymnasium walker state,
2. creates several valid 4-motor action candidates,
3. asks Jev to choose the best candidate,
4. applies that action for a short control interval,
5. feeds progress, contacts, velocity and recent reward into the next decision.

The reference gait is a **proposal generator and API-failure fallback**. Jev selects the macro-action during normal operation.


In [ ]:
#@title 1. Install dependencies
!apt-get update -qq
!apt-get install -y -qq swig > /dev/null
!pip -q install "gymnasium[box2d]" imageio imageio-ffmpeg typesafe-sdk


In [ ]:
#@title 2. Imports, API key and configuration
import os, time, json, getpass
from collections import deque

import numpy as np
import gymnasium as gym
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from IPython.display import Video, display

from typesafe_sdk import Choice, TypeSafeClient

# Prefer Colab Secrets. If unavailable, fall back to a hidden prompt.
def load_typesafe_key():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("TYPESAFE_API_KEY")
    except Exception:
        pass
    if not key:
        key = os.environ.get("TYPESAFE_API_KEY")
    if not key:
        key = getpass.getpass("Enter TYPESAFE_API_KEY from typesafe.ai: ").strip()
    if not key:
        raise RuntimeError("No TYPESAFE_API_KEY supplied.")
    return key

os.environ["TYPESAFE_API_KEY"] = load_typesafe_key()
os.environ.setdefault("TYPESAFE_DEFAULT_MODEL", "jev-latest")

client = TypeSafeClient()

SEED = 0
MAX_STEPS = 800
DECISION_EVERY = 10       # 10 frames ~= 0.20 s simulated time at 50 Hz
MIN_CONFIDENCE = 0.15     # experimental low-risk Gym fallback threshold
VIDEO_EVERY = 4
VIDEO_PATH = "/content/jev_bipedalwalker.mp4"

print("✓ TypeSafe key loaded securely")
print("✓ Jev client ready")
print("model:", os.environ["TYPESAFE_DEFAULT_MODEL"])


In [ ]:
#@title 3. Test the Jev API key
response = client.system_one(
    state={
        "task": "API connectivity test",
        "walker": {"hull_angle": 0.0, "horizontal_speed": 0.1},
    },
    questions={
        "motor_plan": Choice(
            instructions="Choose the better plan for a stable walking robot.",
            criteria={
                "keep_nominal": "Continue the current stable gait.",
                "stabilize": "Prioritize balance recovery before speed.",
            },
        )
    },
)

answer = response.answers["motor_plan"]
print("✓ Jev API works")
print("choice:", answer.choice)
print("confidence:", round(float(answer.confidence), 3))
print("probabilities:", answer.probabilities)
print("served model:", response.model)


In [ ]:
#@title 4. Reference gait + Jev action candidates
class ReferenceWalkerController:
    STAY_ON_ONE_LEG, PUT_OTHER_DOWN, PUSH_OFF = 1, 2, 3
    SPEED = 0.29
    SUPPORT_KNEE_ANGLE = 0.1

    def __init__(self):
        self.state = self.STAY_ON_ONE_LEG
        self.moving_leg = 0
        self.supporting_leg = 1
        self.supporting_knee_angle = self.SUPPORT_KNEE_ANGLE
        self.a = np.zeros(4, dtype=np.float32)

    def step(self, s):
        moving_s_base = 4 + 5 * self.moving_leg
        supporting_s_base = 4 + 5 * self.supporting_leg

        hip_targ = [None, None]
        knee_targ = [None, None]
        hip_todo = [0.0, 0.0]
        knee_todo = [0.0, 0.0]

        if self.state == self.STAY_ON_ONE_LEG:
            hip_targ[self.moving_leg] = 1.1
            knee_targ[self.moving_leg] = -0.6
            self.supporting_knee_angle += 0.03
            if s[2] > self.SPEED:
                self.supporting_knee_angle += 0.03
            self.supporting_knee_angle = min(
                self.supporting_knee_angle, self.SUPPORT_KNEE_ANGLE
            )
            knee_targ[self.supporting_leg] = self.supporting_knee_angle
            if s[supporting_s_base] < 0.10:
                self.state = self.PUT_OTHER_DOWN

        if self.state == self.PUT_OTHER_DOWN:
            hip_targ[self.moving_leg] = 0.1
            knee_targ[self.moving_leg] = self.SUPPORT_KNEE_ANGLE
            knee_targ[self.supporting_leg] = self.supporting_knee_angle
            if s[moving_s_base + 4]:
                self.state = self.PUSH_OFF
                self.supporting_knee_angle = min(
                    s[moving_s_base + 2], self.SUPPORT_KNEE_ANGLE
                )

        if self.state == self.PUSH_OFF:
            knee_targ[self.moving_leg] = self.supporting_knee_angle
            knee_targ[self.supporting_leg] = 1.0
            if s[supporting_s_base + 2] > 0.88 or s[2] > 1.2 * self.SPEED:
                self.state = self.STAY_ON_ONE_LEG
                self.moving_leg = 1 - self.moving_leg
                self.supporting_leg = 1 - self.moving_leg

        if hip_targ[0] is not None:
            hip_todo[0] = 0.9 * (hip_targ[0] - s[4]) - 0.25 * s[5]
        if hip_targ[1] is not None:
            hip_todo[1] = 0.9 * (hip_targ[1] - s[9]) - 0.25 * s[10]
        if knee_targ[0] is not None:
            knee_todo[0] = 4.0 * (knee_targ[0] - s[6]) - 0.25 * s[7]
        if knee_targ[1] is not None:
            knee_todo[1] = 4.0 * (knee_targ[1] - s[11]) - 0.25 * s[12]

        # Hull and vertical damping.
        hip_todo[0] -= 0.9 * (0.0 - s[0]) - 1.5 * s[1]
        hip_todo[1] -= 0.9 * (0.0 - s[0]) - 1.5 * s[1]
        knee_todo[0] -= 15.0 * s[3]
        knee_todo[1] -= 15.0 * s[3]

        self.a[:] = [hip_todo[0], knee_todo[0], hip_todo[1], knee_todo[1]]
        return np.clip(0.5 * self.a, -1.0, 1.0).astype(np.float32)


def clip_action(a):
    return np.clip(np.asarray(a, dtype=np.float32), -1.0, 1.0)


def action_text(a):
    a = clip_action(a)
    return (
        f"[leg0_hip={a[0]:+.3f}, leg0_knee={a[1]:+.3f}, "
        f"leg1_hip={a[2]:+.3f}, leg1_knee={a[3]:+.3f}]"
    )


def build_candidates(obs, reference_action, previous_action):
    ref = clip_action(reference_action)
    prev = clip_action(previous_action)

    balance_term = float(np.clip(0.70 * obs[0] + 1.00 * obs[1], -0.35, 0.35))
    balance = ref.copy()
    balance[[0, 2]] = np.clip(balance[[0, 2]] + balance_term, -1.0, 1.0)

    knee_soft = ref.copy()
    knee_soft[[1, 3]] *= 0.55

    knee_push = ref.copy()
    knee_push[[1, 3]] = np.clip(1.20 * knee_push[[1, 3]], -1.0, 1.0)

    candidates = {
        "nominal": ref,
        "gentle": clip_action(0.65 * ref),
        "assertive": clip_action(1.20 * ref),
        "balance": balance,
        "knee_soft": knee_soft,
        "knee_push": knee_push,
        "smooth": clip_action(0.55 * prev + 0.45 * ref),
        "hold_previous": prev,
        "neutral": np.zeros(4, dtype=np.float32),
    }

    criteria = {
        "nominal": "Reference gait unchanged. Best default when stable. Motors " + action_text(candidates["nominal"]),
        "gentle": "Lower torque for excessive motion or energy use. Motors " + action_text(candidates["gentle"]),
        "assertive": "Stronger gait when forward progress is weak but balance is acceptable. Motors " + action_text(candidates["assertive"]),
        "balance": "Emphasize hull stabilization when tilt/angular velocity is the main risk. Motors " + action_text(candidates["balance"]),
        "knee_soft": "Reduce knee authority when vertical oscillation or overextension is a concern. Motors " + action_text(candidates["knee_soft"]),
        "knee_push": "Increase knee authority for stronger push-off. Motors " + action_text(candidates["knee_push"]),
        "smooth": "Blend previous and reference commands to avoid abrupt switching. Motors " + action_text(candidates["smooth"]),
        "hold_previous": "Hold the previous command when already stable. Motors " + action_text(candidates["hold_previous"]),
        "neutral": "Zero torque only when active commands appear more dangerous than coasting. Motors " + action_text(candidates["neutral"]),
    }
    return candidates, criteria

print("✓ controller and candidate generator ready")


In [ ]:
#@title 5. Convert Gym state to Jev state
def walker_state_for_jev(obs, step, total_reward, recent_rewards, previous_action, reference_action):
    r = list(recent_rewards)
    return {
        "task": {
            "environment": "Gymnasium BipedalWalker-v3",
            "goal": (
                "Move right while staying upright. Avoid hull-ground contact. "
                "Prefer smooth alternating gait and efficient torque."
            ),
            "decision_horizon_frames": DECISION_EVERY,
        },
        "progress": {
            "step": int(step),
            "total_reward": round(float(total_reward), 3),
            "recent_reward_sum": round(float(sum(r)), 3),
            "recent_reward_mean": round(float(np.mean(r)) if r else 0.0, 4),
        },
        "hull": {
            "angle_rad": round(float(obs[0]), 4),
            "angular_velocity_scaled": round(float(obs[1]), 4),
            "horizontal_speed_scaled": round(float(obs[2]), 4),
            "vertical_speed_scaled": round(float(obs[3]), 4),
        },
        "leg0": {
            "hip_angle": round(float(obs[4]), 4),
            "hip_speed": round(float(obs[5]), 4),
            "knee_angle_shifted": round(float(obs[6]), 4),
            "knee_speed": round(float(obs[7]), 4),
            "ground_contact": bool(obs[8] > 0.5),
        },
        "leg1": {
            "hip_angle": round(float(obs[9]), 4),
            "hip_speed": round(float(obs[10]), 4),
            "knee_angle_shifted": round(float(obs[11]), 4),
            "knee_speed": round(float(obs[12]), 4),
            "ground_contact": bool(obs[13] > 0.5),
        },
        "terrain": {
            "lidar_fractions": [round(float(x), 3) for x in obs[14:24]],
            "closest_fraction": round(float(np.min(obs[14:24])), 3),
        },
        "control": {
            "previous_action": [round(float(x), 3) for x in previous_action],
            "reference_action": [round(float(x), 3) for x in reference_action],
        },
    }


In [ ]:
#@title 6. Run one Jev-controlled episode
env = gym.make("BipedalWalker-v3", render_mode="rgb_array")
obs, info = env.reset(seed=SEED)

reference = ReferenceWalkerController()
previous_action = np.zeros(4, dtype=np.float32)
current_action = previous_action.copy()

recent_rewards = deque(maxlen=max(DECISION_EVERY * 2, 20))
total_reward = 0.0
decision_log = []
reward_history = []
speed_history = []
fallbacks = 0
api_failures = 0

writer = imageio.get_writer(VIDEO_PATH, fps=max(1, 50 // VIDEO_EVERY))

try:
    for step in range(MAX_STEPS):
        reference_action = reference.step(obs)

        if step % DECISION_EVERY == 0:
            candidates, criteria = build_candidates(obs, reference_action, previous_action)
            state = walker_state_for_jev(
                obs, step, total_reward, recent_rewards, previous_action, reference_action
            )

            t0 = time.perf_counter()
            try:
                response = client.system_one(
                    state=state,
                    questions={
                        "motor_plan": Choice(
                            instructions=(
                                "Choose the single best motor plan for the next short interval. "
                                "Move right, keep the hull upright, avoid falling, maintain an "
                                "alternating gait, and avoid unnecessary torque. Prefer nominal "
                                "when the walker is already stable."
                            ),
                            criteria=criteria,
                        )
                    },
                )
                latency_ms = (time.perf_counter() - t0) * 1000.0
                answer = response.answers["motor_plan"]

                selected = answer.choice
                confidence = float(answer.confidence)

                if selected not in candidates:
                    raise RuntimeError(f"Unknown Jev choice: {selected}")

                used_fallback = False
                if confidence < MIN_CONFIDENCE:
                    selected = "nominal"
                    used_fallback = True
                    fallbacks += 1

                current_action = candidates[selected].copy()

                decision_log.append({
                    "step": step,
                    "choice": selected,
                    "confidence": confidence,
                    "probabilities": dict(answer.probabilities),
                    "latency_ms": latency_ms,
                    "model": response.model,
                    "fallback": used_fallback,
                    "action": current_action.tolist(),
                    "horizontal_speed": float(obs[2]),
                    "hull_angle": float(obs[0]),
                    "total_reward": float(total_reward),
                })

                if len(decision_log) <= 5 or len(decision_log) % 10 == 0:
                    print(
                        f"decision {len(decision_log):03d} | step {step:04d} | "
                        f"{selected:>13s} | conf {confidence:.3f} | "
                        f"{latency_ms:.0f} ms | vx {obs[2]:+.3f}"
                        + (" | FALLBACK" if used_fallback else "")
                    )

            except Exception as e:
                api_failures += 1
                fallbacks += 1
                current_action = reference_action.copy()
                decision_log.append({
                    "step": step,
                    "choice": "api_error_reference_fallback",
                    "confidence": 0.0,
                    "latency_ms": float("nan"),
                    "model": None,
                    "fallback": True,
                    "action": current_action.tolist(),
                    "total_reward": float(total_reward),
                    "error": repr(e),
                })
                print(f"step {step}: Jev error -> reference fallback: {e}")

            previous_action = current_action.copy()

        obs, reward, terminated, truncated, info = env.step(current_action)
        total_reward += float(reward)
        recent_rewards.append(float(reward))
        reward_history.append(total_reward)
        speed_history.append(float(obs[2]))

        if step % VIDEO_EVERY == 0:
            writer.append_data(env.render())

        if terminated or truncated:
            print(f"Episode ended at step {step + 1}")
            break

finally:
    writer.close()
    env.close()

print("\n=== RESULT ===")
print("steps:", len(reward_history))
print("total reward:", round(total_reward, 2))
print("Jev decisions:", len(decision_log))
print("fallbacks:", fallbacks)
print("API failures:", api_failures)

lat = [d["latency_ms"] for d in decision_log if np.isfinite(d.get("latency_ms", np.nan))]
if lat:
    print("mean Jev latency (ms):", round(float(np.mean(lat)), 1))
    print("p95 Jev latency (ms):", round(float(np.percentile(lat, 95)), 1))

display(Video(VIDEO_PATH, embed=True))


In [ ]:
#@title 7. Plot results and save Jev decision log
plt.figure(figsize=(12, 4))
plt.plot(reward_history)
plt.xlabel("Environment step")
plt.ylabel("Cumulative reward")
plt.title("Jev-controlled BipedalWalker — cumulative reward")
plt.grid(True, alpha=0.25)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(speed_history)
plt.axhline(0.29, linestyle="--", linewidth=1, label="reference target speed")
plt.xlabel("Environment step")
plt.ylabel("Scaled horizontal speed")
plt.title("Forward speed")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

if decision_log:
    plt.figure(figsize=(12, 4))
    plt.plot(
        [d["step"] for d in decision_log],
        [d.get("confidence", 0.0) for d in decision_log],
        marker="o",
        markersize=3,
    )
    plt.axhline(MIN_CONFIDENCE, linestyle="--", linewidth=1, label="fallback threshold")
    plt.ylim(-0.02, 1.02)
    plt.xlabel("Environment step")
    plt.ylabel("Jev confidence")
    plt.title("Jev control confidence")
    plt.legend()
    plt.grid(True, alpha=0.25)
    plt.show()

LOG_PATH = "/content/jev_bipedalwalker_decisions.json"
with open(LOG_PATH, "w", encoding="utf-8") as f:
    json.dump(decision_log, f, indent=2)

print("saved:", LOG_PATH)


## Tuning

For a quick inexpensive run, set `MAX_STEPS = 300` and `DECISION_EVERY = 12`.

For tighter Jev control, try `DECISION_EVERY = 4` to `6`. This makes more API requests, so it increases wall-clock latency and API usage.

The notebook logs the served Jev model version, confidence, full choice distribution, selected motor action, reward and API latency for later comparison with the local Qwen/JEV controller.
